# Sequence Packing with Document Masks — Solution

**Difficulty**: 🟡 Medium

### Problem Statement

Training data comes as documents of wildly different lengths, but accelerators want rectangular
batches. Padding every document to `max_len` burns compute on padding tokens: Krell et al. (2021)
found that up to 50% of all tokens are padding in common NLP datasets. **Sequence packing** puts
several documents side by side in each row instead. Done naively, the packed batch is a
*different* training problem: tokens attend across document boundaries ("cross-contamination"),
positions keep counting from the start of the row, and the last token of one document is trained
to predict the first token of the next.

Done right, packing is exact — every token sees exactly what it would see if its document were
alone. That takes four ingredients:

1. a **packing** of documents into rows (here: first-fit-decreasing bin packing) plus `doc_ids`
   recording which document fills each slot;
2. **position ids** that restart at 0 for every document;
3. **labels** that are −100 wherever the next token belongs to another document or is padding;
4. an **attention mask** that is block-diagonal *and* causal — or, for FlashAttention's
   variable-length kernels, **`cu_seqlens`**, the cumulative document lengths that let the kernel
   attend within each document and never compute the cross-document blocks at all.

```
max_len = 8, seqs = [a (5 tokens), b (3), c (4), d (2)]   ->   FFD visits a, c, b, d

              input_ids                   doc_ids                 position_ids
row 0:  a1 a2 a3 a4 a5 b1 b2 b3     0  0  0  0  0  1  1  1     0 1 2 3 4 0 1 2
row 1:  c1 c2 c3 c4 d1 d2 ·  ·      2  2  2  2  3  3 -1 -1     0 1 2 3 0 1 0 0

labels  row 0:  a2 a3 a4 a5 -100 b2 b3 -100            cu_seqlens = [0, 5, 8, 12, 14]  (int32)
        row 1:  c2 c3 c4 -100 d2 -100 -100 -100

mask, row 1 (query ↓, key →; ✓ = may attend):
        c1 c2 c3 c4 d1 d2 ·  ·
   c1   ✓
   c2   ✓  ✓
   c3   ✓  ✓  ✓
   c4   ✓  ✓  ✓  ✓
   d1               ✓
   d2               ✓  ✓
   ·                      ✓
   ·                      ✓  ✓
```

---

### Requirements

1. **`pack_sequences(seqs, max_len, pad_id=0) -> (input_ids, position_ids, doc_ids, cu_seqlens)`**
   - `seqs`: list of 1-D integer tensors with $1 \le$ `len` $\le$ `max_len`; anything else (an
     empty or too-long sequence) → `ValueError`.
   - **First-fit decreasing, deterministic:** visit the sequences in the order
     `sorted(range(n), key=lambda i: (-len(seqs[i]), i))` (longest first, equal lengths by
     index). Put each into the **lowest-index** row whose free space is at least its length; if
     no row fits, open a new row at the end. (The tie-break among equal lengths is only a
     convention for a reproducible layout: the grader accepts any order among equal lengths,
     since it never changes which lengths share a row. Note:
     `torch.argsort(lengths, descending=True)` keeps equal lengths in index order only with
     `stable=True`.)
   - **Layout:** a row's documents sit left to right in the order they were placed, starting at
     column 0 with no gaps; the rest of the row is padding.
   - `input_ids`: `(R, max_len)` int64, `pad_id` at padding.
   - `position_ids`: `(R, max_len)` int64, $0, 1, \dots, n-1$ inside every document, 0 at padding.
   - `doc_ids`: `(R, max_len)` int64, the document's index in `seqs`, −1 at padding.
   - `cu_seqlens`: `(num_docs + 1,)` **int32** in the `flash_attn_varlen_func` convention
     ("the cumulative sequence lengths of the sequences in the batch, used to index into q",
     with `q` of shape `(total_q, nheads, headdim)`). It indexes the **unpadded, flattened**
     token stream `input_ids[doc_ids >= 0]` (real tokens in row-major order), one segment per
     **document**: `cu_seqlens[0] = 0`, segment $k$ = `[cu_seqlens[k], cu_seqlens[k+1])` is the
     $k$-th document in that order (not necessarily `seqs[k]`), `cu_seqlens[-1]` = number of real
     tokens. (The kernel also wants `max_seqlen`, the largest segment length.)
   - No sequences → `(0, max_len)` tensors and `cu_seqlens = tensor([0], dtype=torch.int32)`.
   - Padding is a property of the layout: never infer it from token values (`pad_id` may also
     be a real token).
2. **`packed_labels(input_ids, doc_ids) -> labels`** — `(R, L)` int64, **already shifted**:
   `labels[r, t]` is the target of the logits at position $t$, so the loss is
   `F.cross_entropy(logits.flatten(0, 1), labels.flatten())` (its default `ignore_index` is −100).
   - `labels[r, t] = input_ids[r, t + 1]` if $t + 1 < L$ and
     `doc_ids[r, t + 1] == doc_ids[r, t] != -1`; otherwise −100.
   - So −100 at the last token of every document, on every padding position and in the last
     column. Must work for any `doc_ids`, not only the output of `pack_sequences`.
   - (Hugging Face `*ForCausalLM` models take *unshifted* labels and shift inside the loss; their
     `DataCollatorWithFlattening` gets the same effect by writing −100 at the *first* token of
     every packed sequence. `v4/multimodal/vlm-connector` uses that unshifted convention; this
     problem does not.)
3. **`document_causal_mask(doc_ids) -> mask`** — `(R, L, L)` bool, **True = may attend** (the
   boolean `attn_mask` convention of `F.scaled_dot_product_attention`):
   - `mask[r, i, j] = (doc_ids[r, i] == doc_ids[r, j]) and (j <= i)`.
   - Padding counts as one more document (id −1): a padding query attends to the padding keys at
     or before it in its row. Every row therefore contains at least its diagonal, so no softmax
     row is all $-\infty$ (which would be $0/0 = $ NaN), and no real token ever attends to padding.
4. **The property that matters:** the small transformer given below (learned position
   embeddings, one pre-LN block) run on the packed batch with your `position_ids`, mask and
   labels reproduces — token for token — the logits and next-token losses of every document run
   on its own. Test 5 and the grader check exactly that.

---

### Constraints

- ✅ Python loops are fine for the bin packing itself (it runs in the data loader, over lengths).
- ✅ `packed_labels` and `document_causal_mask` should be vectorized: slicing, comparisons,
  broadcasting, `torch.tril`, `torch.where` (not timed by the grader, but an interviewer will ask).
- ❌ No `flash_attn` or `flex_attention` needed — the notebook gives you a reference varlen
  attention to test `cu_seqlens` with.
- ❌ Don't decide what is padding from token values.

---

<details>
  <summary>💡 Hint</summary>

  - First-fit decreasing needs two pieces of state while it places documents. Which?
  - After packing, everything is a function of `doc_ids`. How do you ask "is the next position
    in the same document?" for every position at once?
  - Draw the mask for one row holding two documents: it is the AND of two simple patterns.
  - What does a softmax do on a row that is entirely $-\infty$?

</details>

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
IGNORE_INDEX = -100


def make_docs(lengths, vocab=50, seed=0):
    """Random token documents (ids >= 1, so 0 is free to be the pad id)."""
    g = torch.Generator().manual_seed(seed)
    return [torch.randint(1, vocab, (n,), generator=g) for n in lengths]


class TinyLM(nn.Module):
    """Token + LEARNED absolute position embeddings, one pre-LN block, LM head.

    forward(input_ids (R, L), position_ids (R, L), mask (R, L, L) bool, True = may attend)
    -> logits (R, L, vocab). Attention is written out by hand: a fully masked row gives NaN.
    """

    def __init__(self, vocab, max_len, d=16, n_heads=2):
        super().__init__()
        self.h = n_heads
        self.tok, self.pos = nn.Embedding(vocab, d), nn.Embedding(max_len, d)
        self.ln1, self.ln2, self.ln_f = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv, self.proj = nn.Linear(d, 3 * d), nn.Linear(d, d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.head = nn.Linear(d, vocab)

    def forward(self, input_ids, position_ids, mask):
        R, L = input_ids.shape
        x = self.tok(input_ids) + self.pos(position_ids)
        q, k, v = self.qkv(self.ln1(x)).chunk(3, dim=-1)
        q, k, v = (t.view(R, L, self.h, -1).transpose(1, 2) for t in (q, k, v))
        scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
        attn = scores.masked_fill(~mask[:, None], float("-inf")).softmax(dim=-1)
        x = x + self.proj((attn @ v).transpose(1, 2).reshape(R, L, -1))
        x = x + self.mlp(self.ln2(x))
        return self.head(self.ln_f(x))


def varlen_causal_attention(q, k, v, cu_seqlens):
    """What flash_attn_varlen_func(q, k, v, cu, cu, max_len, max_len, causal=True) computes.

    q, k, v: (total_tokens, n_heads, head_dim) — the UNPADDED tokens of the whole batch.
    Each segment [cu_seqlens[i], cu_seqlens[i+1]) is attended to on its own.
    """
    out = torch.empty_like(q)
    for s, e in zip(cu_seqlens[:-1].tolist(), cu_seqlens[1:].tolist()):
        qs, ks, vs = (t[s:e].transpose(0, 1) for t in (q, k, v))           # (H, n, d_h)
        out[s:e] = F.scaled_dot_product_attention(qs, ks, vs, is_causal=True).transpose(0, 1)
    return out

In [ ]:
def pack_sequences(seqs, max_len, pad_id=0):
    """First-fit-decreasing packing of variable-length documents into rows of length max_len.

    Args:
        seqs: list of 1-D int64 tensors, each with 1 <= len <= max_len (else ValueError).
        max_len: row length L.
        pad_id: token id written into padding slots.

    Returns:
        input_ids:    (R, L) int64, pad_id on padding.
        position_ids: (R, L) int64, 0..n-1 inside each document, 0 on padding.
        doc_ids:      (R, L) int64, index into seqs, -1 on padding.
        cu_seqlens:   (num_docs + 1,) int32, [0, cumulative document lengths] over the unpadded
                      tokens input_ids[doc_ids >= 0] in row-major order.
    """
    lengths = [int(s.numel()) for s in seqs]
    for i, n in enumerate(lengths):
        if not 1 <= n <= max_len:
            raise ValueError(f"sequence {i} has length {n}; need 1 <= length <= max_len={max_len}")

    # First-fit decreasing: longest first (ties: lower index), into the lowest-index row that fits.
    order = sorted(range(len(seqs)), key=lambda i: (-lengths[i], i))
    rows, free = [], []                       # document indices per row, free slots per row
    for i in order:
        for r in range(len(rows)):
            if free[r] >= lengths[i]:
                rows[r].append(i)
                free[r] -= lengths[i]
                break
        else:                                 # no row had room: open a new one
            rows.append([i])
            free.append(max_len - lengths[i])

    R = len(rows)
    input_ids = torch.full((R, max_len), pad_id, dtype=torch.long)
    position_ids = torch.zeros(R, max_len, dtype=torch.long)
    doc_ids = torch.full((R, max_len), -1, dtype=torch.long)
    seg_lens = []                             # document lengths in row-major order
    for r, docs in enumerate(rows):
        t = 0
        for i in docs:
            n = lengths[i]
            input_ids[r, t:t + n] = seqs[i]
            position_ids[r, t:t + n] = torch.arange(n)
            doc_ids[r, t:t + n] = i
            seg_lens.append(n)
            t += n
    cu_seqlens = torch.zeros(len(seg_lens) + 1, dtype=torch.int32)
    cu_seqlens[1:] = torch.tensor(seg_lens, dtype=torch.long).cumsum(0)
    return input_ids, position_ids, doc_ids, cu_seqlens

In [ ]:
def packed_labels(input_ids, doc_ids):
    """Shifted next-token labels that never cross a document boundary.

    Args:
        input_ids: (R, L) int64.
        doc_ids:   (R, L) int64, -1 on padding.

    Returns:
        labels: (R, L) int64; labels[r, t] = input_ids[r, t+1] if position t+1 is in the same
        non-padding document as t, else -100 (last token of each document, padding, last column).
    """
    labels = torch.full_like(input_ids, IGNORE_INDEX)
    same_doc_next = (doc_ids[:, 1:] == doc_ids[:, :-1]) & (doc_ids[:, :-1] >= 0)
    labels[:, :-1] = torch.where(same_doc_next, input_ids[:, 1:], IGNORE_INDEX)
    return labels


def document_causal_mask(doc_ids):
    """Block-diagonal causal attention mask for a packed batch.

    Args:
        doc_ids: (R, L) int64, -1 on padding (padding counts as one more document).

    Returns:
        mask: (R, L, L) bool, mask[r, i, j] = (doc_ids[r, i] == doc_ids[r, j]) & (j <= i);
        True = query i may attend to key j.
    """
    L = doc_ids.shape[-1]
    same_doc = doc_ids[:, :, None] == doc_ids[:, None, :]           # (R, L, L)
    causal = torch.ones(L, L, dtype=torch.bool, device=doc_ids.device).tril()
    return same_doc & causal                  # -1 == -1 keeps padding rows non-empty: no NaN

In [ ]:
# Test 1: the worked example from the problem statement
a, b, c, d = (torch.tensor(t) for t in ([11, 12, 13, 14, 15], [21, 22, 23], [31, 32, 33, 34], [41, 42]))
ids, pos, doc, cu = pack_sequences([a, b, c, d], max_len=8, pad_id=0)
assert ids.tolist() == [[11, 12, 13, 14, 15, 21, 22, 23], [31, 32, 33, 34, 41, 42, 0, 0]], ids
assert doc.tolist() == [[0, 0, 0, 0, 0, 1, 1, 1], [2, 2, 2, 2, 3, 3, -1, -1]], doc
assert pos.tolist() == [[0, 1, 2, 3, 4, 0, 1, 2], [0, 1, 2, 3, 0, 1, 0, 0]], pos
assert cu.dtype == torch.int32 and cu.tolist() == [0, 5, 8, 12, 14], cu

labels = packed_labels(ids, doc)
assert labels.tolist() == [[12, 13, 14, 15, -100, 22, 23, -100],
                           [32, 33, 34, -100, 42, -100, -100, -100]], labels

mask = document_causal_mask(doc)
assert mask.dtype == torch.bool and tuple(mask.shape) == (2, 8, 8)
want_row1 = torch.tensor([[1, 0, 0, 0, 0, 0, 0, 0],
                          [1, 1, 0, 0, 0, 0, 0, 0],
                          [1, 1, 1, 0, 0, 0, 0, 0],
                          [1, 1, 1, 1, 0, 0, 0, 0],
                          [0, 0, 0, 0, 1, 0, 0, 0],
                          [0, 0, 0, 0, 1, 1, 0, 0],
                          [0, 0, 0, 0, 0, 0, 1, 0],
                          [0, 0, 0, 0, 0, 0, 1, 1]], dtype=torch.bool)
assert torch.equal(mask[1], want_row1), mask[1].int()
print(mask[1].int())
print("Test 1 passed")

In [ ]:
# Test 2: a bigger packing — every document exactly once, contiguous, positions reset
lengths = [7, 3, 12, 5, 9, 2, 4, 6, 1, 16, 8, 11, 3, 3]
MAX_LEN = 16
docs = make_docs(lengths, seed=1)
ids, pos, doc, cu = pack_sequences(docs, max_len=MAX_LEN)
for i, dd in enumerate(docs):
    where = (doc == i).nonzero()
    assert len(where) == len(dd) and where[:, 0].unique().numel() == 1, f"doc {i} must sit whole in one row"
    r, cols = where[0, 0], where[:, 1]
    assert torch.equal(ids[r, cols], dd), f"doc {i}'s tokens are wrong"
    assert torch.equal(pos[r, cols], torch.arange(len(dd))), f"doc {i}'s positions must restart at 0"
flat = ids[doc >= 0]                                   # the unpadded token stream
segments = [flat[s:e].tolist() for s, e in zip(cu[:-1].tolist(), cu[1:].tolist())]
assert sorted(segments) == sorted(dd.tolist() for dd in docs), "cu_seqlens must cut out whole documents"
print(f"{len(docs)} docs, {sum(lengths)} tokens -> {ids.shape[0]} rows of {MAX_LEN} "
      f"(lower bound: ceil({sum(lengths)}/{MAX_LEN}) = {math.ceil(sum(lengths) / MAX_LEN)})")

# FFD is a heuristic: 3 rows here, although {4, 3, 3} twice would fit in 2
assert pack_sequences(make_docs([4, 4, 3, 3, 3, 3]), 10)[0].shape[0] == 3
for bad in ([torch.arange(9)], [torch.arange(3), torch.zeros(0, dtype=torch.long)]):
    try:
        pack_sequences(bad, 8)
        raise AssertionError("expected ValueError")
    except ValueError:
        pass
print("Test 2 passed")

In [ ]:
# Test 3: labels and mask against brute force (packed rows plus hand-made doc_ids)
def labels_brute(ids, doc):
    out = torch.full_like(ids, IGNORE_INDEX)
    for r in range(ids.shape[0]):
        for t in range(ids.shape[1] - 1):
            if doc[r, t] != -1 and doc[r, t + 1] == doc[r, t]:
                out[r, t] = ids[r, t + 1]
    return out

def mask_brute(doc):
    R, L = doc.shape
    m = torch.zeros(R, L, L, dtype=torch.bool)
    for r in range(R):
        for i in range(L):
            for j in range(i + 1):
                m[r, i, j] = bool(doc[r, i] == doc[r, j])
    return m

hand_doc = torch.tensor([[5, 5, 2, 2, 2, -1], [0, 1, 1, 1, 3, 3], [-1, -1, -1, -1, -1, -1]])
hand_ids = torch.randint(1, 9, hand_doc.shape)
for i_, d_ in ((ids, doc), (hand_ids, hand_doc)):
    assert torch.equal(packed_labels(i_, d_), labels_brute(i_, d_))
    m_ = document_causal_mask(d_)
    assert torch.equal(m_, mask_brute(d_))
    assert m_.any(-1).all(), "every query must be allowed to attend somewhere (else softmax -> NaN)"
print("Test 3 passed")

In [ ]:
# Test 4: cu_seqlens drives a varlen kernel — same result as dense attention with your mask
torch.manual_seed(0)
H, D_H = 2, 4
mask = document_causal_mask(doc)
q, k, v = (torch.randn(*ids.shape, H, D_H, dtype=torch.float64) for _ in range(3))   # (R, L, H, d_h)
dense = F.scaled_dot_product_attention(*(t.transpose(1, 2) for t in (q, k, v)),
                                       attn_mask=mask[:, None]).transpose(1, 2)
real = doc >= 0
varlen = varlen_causal_attention(q[real], k[real], v[real], cu)                      # (total, H, d_h)
assert torch.allclose(dense[real], varlen, atol=1e-12), "masked dense attention != varlen attention"
seg = cu[1:] - cu[:-1]
print(f"dense masked attention scores: {ids.shape[0] * MAX_LEN ** 2}, "
      f"varlen (causal, per document): {int((seg * (seg + 1) // 2).sum())}")
print("Test 4 passed")

In [ ]:
# Test 5: the packed batch is exactly equivalent to running every document alone
torch.manual_seed(0)
model = TinyLM(vocab=50, max_len=MAX_LEN).double()
mask, labels = document_causal_mask(doc), packed_labels(ids, doc)
logits = model(ids, pos, mask)
assert torch.isfinite(logits).all(), "NaN/inf logits: some attention row is fully masked"

total, count = 0.0, 0
for i, dd in enumerate(docs):
    n = len(dd)
    alone = model(dd[None], torch.arange(n)[None], torch.ones(n, n, dtype=torch.bool).tril()[None])[0]
    err = (logits[doc == i] - alone).abs().max().item()
    assert err < 1e-10, f"doc {i}: packed logits differ from the document alone (max err {err:.2e})"
    total += F.cross_entropy(alone[:-1], dd[1:], reduction="sum").item()
    count += n - 1
packed_loss = F.cross_entropy(logits.flatten(0, 1), labels.flatten()).item()
assert abs(packed_loss - total / count) < 1e-10, (packed_loss, total / count)
print(f"packed mean loss {packed_loss:.6f} == token-weighted mean over documents {total / count:.6f}")

# What goes wrong without the document mask / without resetting positions
R, L = ids.shape
causal_only = torch.ones(L, L, dtype=torch.bool).tril().expand(R, L, L)
row_pos = torch.arange(L).expand(R, L)
print("max logit error, causal-only mask:  ", (model(ids, pos, causal_only) - logits)[real].abs().max().item())
print("max logit error, positions not reset:", (model(ids, row_pos, mask) - logits)[real].abs().max().item())

# Throughput: how much of the compute is padding?
g = torch.Generator().manual_seed(0)
lens = torch.exp(4.0 + 0.9 * torch.randn(500, generator=g)).long().clamp(1, 512)   # long-tailed
ids_p, _, doc_p, _ = pack_sequences(make_docs(lens.tolist(), seed=2), max_len=512)
batch_max = lens[torch.randperm(500, generator=g)].view(-1, 10).max(1).values        # batches of 10
real_tokens = lens.sum().item()
print(f"real tokens {real_tokens}: useful fraction "
      f"pad-to-512 {real_tokens / (500 * 512):.1%}, "
      f"pad-to-longest-in-batch {real_tokens / (10 * batch_max.sum().item()):.1%}, "
      f"packed into {ids_p.shape[0]} rows {real_tokens / ids_p.numel():.1%}")
print("Test 5 passed")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("sequence-packing", pack_sequences=pack_sequences, packed_labels=packed_labels,
      document_causal_mask=document_causal_mask)

### Why it works

Every operation in a transformer except attention acts on each token independently (embeddings,
LayerNorm, MLP, LM head). The packed and separate forwards can therefore only differ through what
attention lets a token see and which position each token is given.

- **Mask.** The block-diagonal causal mask lets token $i$ of document $A$ attend to exactly
  tokens $0..i$ of $A$ — the same set as $A$'s own causal mask. Every other key gets $-\infty$,
  i.e. weight exactly $e^{-\infty} = 0$, so even the softmax normaliser is unchanged.
- **Positions.** Position ids that restart at 0 give every token the position embedding it would
  have had alone.
- **Labels.** Each token gets the same target as alone, and the cross-boundary pairs (last token
  of $A$ → first token of $B$) get −100, which `F.cross_entropy` drops from both the sum and the
  count. The packed sum of token losses therefore equals the sum over documents, and the packed
  mean is the token-weighted mean over all documents.
- **Padding** is its own causal block (document −1): its outputs are finite junk, its labels are
  −100, and no real token attends to it, so it changes neither the loss nor the gradients.

**First-fit decreasing.** Placing large documents first lets the small ones fill the gaps.
Bin packing is NP-hard and FFD is only a heuristic — on lengths $[4, 4, 3, 3, 3, 3]$ with
`max_len = 10` it uses 3 rows although $\{4, 3, 3\}$ twice fits in 2 — but it is guaranteed to use
at most $\frac{11}{9}\mathrm{OPT} + \frac{6}{9}$ rows (Dósa, 2007).

### Complexity

- Packing: $O(n \log n)$ to sort plus $O(nR)$ for the first-fit scan ($O(n \log R)$ with a
  segment tree over the rows' free space); building the outputs is $O(RL)$.
- Labels: $O(RL)$. Dense mask: $O(RL^2)$ memory, and masked dense attention still computes all
  $RL^2$ scores; a varlen kernel computes $\sum_i n_i(n_i+1)/2$ causal scores and needs only the
  $O(\text{num\_docs})$ `cu_seqlens` (Test 4 prints both).

### Common bugs

- **Causal-only mask** — tokens attend to earlier documents in the row (cross-contamination).
- **Position ids from `arange(L)`** over the whole row instead of per document.
- **Labels that cross boundaries**: the last token of a document is trained on the next
  document's first token; `torch.roll` additionally wraps the first token of a row into the last
  column. Mixing up shifted and unshifted label conventions shifts every target by one.
- **`cu_seqlens` without the leading 0**, as int64, one segment per *row* (plain
  `unpad_input` semantics — the documents in a row then attend to each other), or offsets into the
  padded stream.
- **Fully masked padding rows**: `softmax` of an all-$-\infty$ row is $0/0 =$ NaN in a hand-written
  attention, and NaN activations poison the gradients even though those labels are −100.
- **Detecting padding with `input_ids == pad_id`** — real tokens can equal the pad id.
- **Not quite FFD**: unsorted first fit, best fit, or laying out a row's documents in another
  order than they were placed. (An unstable sort only permutes equal-length documents: the
  packing is just as good, but the layout is no longer reproducible.)

### Interview follow-ups

1. **Why pack — how much does it save?** The useful fraction is real tokens / (rows × `max_len`).
   Test 5 prints it for a long-tailed length distribution: padding every document to the maximum
   wastes most of the compute, padding to the longest in a batch helps, packing gets close to
   100%. Krell et al. (2021) report up to 50% padding on common NLP datasets (89% for GLUE-CoLA at
   length 128) and a 2× speed-up for phase-2 BERT pre-training from packing.
2. **Is the document mask necessary? (the cross-contamination debate)** GPT-3 (Brown et al.,
   2020, App. B) packed documents into 2048-token sequences with no special masking, only an
   end-of-text token between documents, "giving the language model the information necessary to
   infer that context separated by the end of text token is unrelated" — simple, fast, and still
   common in pre-training. Llama 3 (2024) masks attention between documents and reports limited
   impact in standard pre-training but importance for continued pre-training on very long
   sequences. For fine-tuning on short examples, each example should condition only on itself,
   so masking (or a varlen kernel) is the norm there.
3. **Dense mask or varlen FlashAttention?** An $(R, L, L)$ mask costs $O(L^2)$ memory per row,
   and a dense kernel still computes the masked blocks. `flash_attn_varlen_func` takes the
   unpadded tokens `(total, H, d_h)` plus `cu_seqlens` and attends within each document only — no
   padding, no wasted blocks; FlexAttention's block mask similarly skips fully masked blocks.
   Hugging Face's padding-free path even derives `cu_seqlens` from where `position_ids` restart
   (`prepare_fa_kwargs_from_position_ids`), so there the position ids double as boundary markers.
4. **How should the loss be normalised after packing?** The packed mean over labelled tokens
   weights every token equally, so long documents dominate; averaging per-document means instead
   weights documents equally. Either is a modelling choice, but it must not depend on how the
   documents happened to be packed. With gradient accumulation, divide the summed loss by the
   number of labelled tokens across *all* micro-batches (Hugging Face passes
   `num_items_in_batch` for this), not by each micro-batch's own count.
5. **With RoPE, do position ids still need to reset?** In exact arithmetic, no: with a correct
   document mask, RoPE scores depend only on the offset $m - n$ inside a document, so shifting all
   of a document's positions cancels (up to floating-point rounding). Learned or absolute
   position embeddings — like this notebook's `TinyLM` — do change, and frameworks that infer
   boundaries from position resets need them. Reset them anyway.
6. **Packing multimodal batches?** Image tokens count toward the length (LLaVA-1.5: a 336-pixel
   image with 14-pixel patches becomes $24 \times 24 = 576$ tokens), so pack by the expanded
   length and never split a sample across rows. Image tokens are never prediction targets: with
   this problem's shifted labels that is −100 wherever the *next* token is an image token
   (`v4/multimodal/vlm-connector`, with unshifted labels, puts −100 on the image positions
   themselves). Models with non-causal parts change the mask inside a
   document — PaliGemma's prefix-LM attends bidirectionally over the image and prompt tokens — so
   the mask becomes "same document AND (causal OR both in that document's prefix)".